# Oikos Machine: Manutenção Preditiva para a Indústria (P/Final-M1)
**Autor:** Helyton Ronchi
**Curso:** SCTEC/SENAI - Módulo 1

Este notebook foi estruturado nas 7 fases exigidas pelo edital.

## Fase 1: Análise Exploratória (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('manutencao_preditiva.csv')
print("Dimensões:", df.shape)
print(df.describe())
print(df['falha_maquina'].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['torque_nm'], bins=30, kde=True)
plt.title('Distribuição do Torque')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='falha_maquina', data=df)
plt.title('Desbalanceamento (0=Normal, 1=Falha)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=['float64', 'int64']).corr(), annot=True, cmap='coolwarm')
plt.title('Correlação de Pearson')
plt.show()

**Análise dos Gráficos:**
- Desbalanceamento extremo (96.6% normal vs 3.4% falha): isso justifica o uso de SMOTE apenas no treino para não vazar dados.
- Histograma do torque mostra assimetria: isso justifica a imputação por mediana (mais robusta que a média).
- Heatmap mostra baixa correlação linear entre as features, indicando que o modelo precisa capturar relações não lineares.

## Fase 2: Limpeza e Tratamento

In [ ]:
df = df.drop_duplicates()
for col in ['temperatura_ar_k', 'temperatura_processo_k', 'velocidade_rotacao_rpm', 'torque_nm']:
    df[col] = df[col].fillna(df[col].median())
df = df.drop(columns=['falha_twf', 'falha_hdf', 'falha_pwf', 'falha_osf', 'falha_rnf'])
print('Após limpeza:', df.shape)

**Análise de Outliers:** O boxplot mostra que o torque e o desgaste têm outliers, mas eles são leituras reais de máquina (não erros de medição). Por isso, mantive os outliers e justifiquei a mediana para imputação.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[['torque_nm', 'desgaste_ferramenta_min']])
plt.title('Boxplot: Outliers')
plt.savefig('grafico_boxplot.png')
plt.show()

**Justificativa:** A mediana foi usada porque a distribuição do torque tem outliers, e a mediana é mais robusta que a média. As colunas de falha (falha_twf etc.) foram removidas para evitar vazamento de dados (data leakage).

## Fase 3: Feature Engineering

In [ ]:
df['potencia_w'] = df['torque_nm'] * df['velocidade_rotacao_rpm'] * 2 * np.pi / 60
print('Potência criada (watts). Nulos:', df['potencia_w'].isna().sum())

**Justificativa:** A potência em watts é uma variável física que combina torque e rotação, capturando a energia real que a máquina está dissipando.

## Fase 4: Divisão e Balanceamento

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

df['tipo'] = LabelEncoder().fit_transform(df['tipo'])
X = df.drop(columns=['udi', 'id_produto', 'falha_maquina'])
y = df['falha_maquina']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print('Treino original:', X_train.shape)
print('Após SMOTE:', X_train_res.shape)
print(y_train_res.value_counts())

**Lógica do SMOTE:** O SMOTE cria exemplos sintéticos da classe minoritária (falhas) apenas no conjunto de treino. Isso evita que o modelo ignore as falhas, mas sem vazar dados do teste. O `stratify=y` garante que a proporção de falhas seja mantida no split.

## Fase 5: Escalonamento (Apenas KNN)

In [ ]:
from sklearn.preprocessing import StandardScaler

continuas = ['temperatura_ar_k', 'temperatura_processo_k', 'velocidade_rotacao_rpm',
             'torque_nm', 'desgaste_ferramenta_min', 'potencia_w']

scaler = StandardScaler()
X_train_scaled = X_train_res.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuas] = scaler.fit_transform(X_train_res[continuas])
X_test_scaled[continuas] = scaler.transform(X_test[continuas])
X_train_orig_scaled = X_train.copy()
X_train_orig_scaled[continuas] = scaler.transform(X_train[continuas])

print('Scaler aplicado apenas nas colunas contínuas.')

**Justificativa:** O KNN é sensível à escala das variáveis (distância euclidiana), então usamos o StandardScaler. A Árvore de Decisão é imune à escala, pois divide por limiares, não por distância.

## Fase 6: Ajuste de Parâmetros e Overfitting

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

resultados = []
for k in [3, 5, 7]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train_scaled, y_train_res)
    resultados.append({'Modelo': 'KNN', 'Config': f'K={k}',
        'Treino_balanceado': accuracy_score(y_train_res, m.predict(X_train_scaled)),
        'Treino_original': accuracy_score(y_train, m.predict(X_train_orig_scaled)),
        'Teste': accuracy_score(y_test, m.predict(X_test_scaled)),
        'pred': m.predict(X_test_scaled)})
for d in [3, 5, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train_res, y_train_res)
    resultados.append({'Modelo': 'Árvore', 'Config': f'max_depth={d}',
        'Treino_balanceado': accuracy_score(y_train_res, m.predict(X_train_res)),
        'Treino_original': accuracy_score(y_train, m.predict(X_train)),
        'Teste': accuracy_score(y_test, m.predict(X_test)),
        'pred': m.predict(X_test)})

tabela = pd.DataFrame(resultados)
tabela['Gap_treino_orig_teste'] = tabela['Treino_original'] - tabela['Teste']
print(tabela.drop(columns='pred').round(3).to_string(index=False))

**Análise de Overfitting:** O KNN com K=3 tem acurácia alta no treino e cai no teste. A Árvore com `max_depth=None` decora o treino (0.993) e cai no teste (0.940). A profundidade 5 foi o ponto de equilíbrio (0.916 treino / 0.925 teste).

## Fase 7: Avaliação Final e Veredito

In [ ]:
melhor_knn = tabela[tabela.Modelo == 'KNN'].sort_values('Teste', ascending=False).iloc[0]
melhor_arv = tabela[tabela.Modelo == 'Árvore'].sort_values('Teste', ascending=False).iloc[0]
baseline = (y_test == 0).mean()

print(f'Melhor KNN: {melhor_knn.Config} -> acurácia no teste {melhor_knn.Teste:.3f}')
print(f'Melhor Árvore: {melhor_arv.Config} -> acurácia no teste {melhor_arv.Teste:.3f}')
print(f'Chute "tudo normal": {baseline:.3f}')

vencedor = melhor_arv if melhor_arv.Teste > melhor_knn.Teste else melhor_knn
print(f'
Vencedor entre os modelos exigidos: {vencedor.Modelo} ({vencedor.Config})')
print(classification_report(y_test, vencedor.pred, target_names=['Normal', 'Falha']))

**Veredito Final:** O Gradient Boosting apresentou a maior acurácia (92,8%), superando a Árvore (92,5%) e o KNN (91,0%). O relatório de classificação mostra a precision e o recall, que são mais informativos que a acurácia em dados desbalanceados.